In [1]:
import logging
from kiteconnect import KiteConnect, KiteTicker
import datetime
logging.basicConfig(level=logging.INFO)
import pandas as pd
import numpy as np
from IPython.display import clear_output

In [ ]:


kite = KiteConnect(api_key=API_KEY)

In [3]:
# may need to re autheticate if expired
REQUEST_TOKEN = "SY4QmvQVMowiLjGeb79pjS2jWodUKnEB"

In [4]:
# Redirect the user to the login url obtained
# from kite.login_url(), and receive the request_token
# from the registered redirect url after the login flow.
# Once you have the request_token, obtain the access_token
# as follows.


data = kite.generate_session(REQUEST_TOKEN, api_secret=API_SCERET)
kite.set_access_token(data["access_token"])

In [5]:
logging.basicConfig(level=logging.DEBUG)

# Initialise
kws = KiteTicker(API_KEY, data["access_token"])

In [6]:
instruments = kite.instruments()

In [7]:
nse_ins = [i for i in instruments if i['exchange'] == "NSE" and i["segment"] in ["NSE", "INDICES"]]
sub_ins = [nse_ins[i]['instrument_token'] for i in range(1000)]

In [8]:
import pickle
from datetime import datetime
from IPython.display import clear_output

ticks = []
BATCH_SIZE = 10000

def dump_ticks(batch):
    fname = f"ticks_{datetime.now().strftime('%Y%m%d_%H%M%S')}.pkl"
    print(f"writing {len(batch)} ticks to {fname}")
    with open(fname, "wb") as f:
        pickle.dump(batch, f)
    print("pickle write complete")

def on_ticks(ws, tick):
    ticks.append(tick)

    clear_output(wait=True)
    print(f"received: {len(ticks)}")

    if len(ticks) >= BATCH_SIZE:
        dump_ticks(ticks)
        ticks.clear()
        print("tick buffer cleared")

def on_connect(ws, response):
    ws.subscribe(sub_ins)
    ws.set_mode(ws.MODE_FULL, sub_ins)

def on_close(ws, code, reason):
    ws.stop()

kws.on_ticks = on_ticks
kws.on_close = on_close
kws.on_connect = on_connect

In [9]:
kws.connect()

received: 7532


ERROR:kiteconnect.ticker:Connection error: 1006 - connection was closed uncleanly (peer dropped the TCP connection without previous WebSocket closing handshake)
ERROR:kiteconnect.ticker:Connection closed: 1006 - connection was closed uncleanly (peer dropped the TCP connection without previous WebSocket closing handshake)


In [10]:
dump_ticks(ticks)

writing 7532 ticks to ticks_20251128_152256.pkl
pickle write complete


In [8]:
len(ticks)

2522

In [11]:
ticks = [i[0] for i in ticks]

In [12]:
ticks_df = pd.DataFrame.from_records(ticks)

In [14]:
pd.DataFrame.from_records(ticks_df.ohlc)

,open,high,low,close
0,158.46,158.68,153.40,158.45
1,158.46,158.68,153.40,158.45
2,158.46,158.68,153.40,158.45
3,158.46,158.68,153.40,158.45
4,158.46,158.68,153.40,158.45
...,...,...,...,...
2517,158.46,158.68,152.62,158.45
2518,158.46,158.68,152.62,158.45
2519,158.46,158.68,152.62,158.45
2520,158.46,158.68,152.50,158.45


In [16]:
ticks_df[:10]

,tradable,mode,instrument_token,last_price,last_traded_quantity,average_traded_price,volume_traded,total_buy_quantity,total_sell_quantity,ohlc,change,last_trade_time,oi,oi_day_high,oi_day_low,exchange_timestamp,depth
0,True,full,194325505,153.69,1,155.10,4776224,682848,1068881,"{'open': 158.46, 'high': 158.68, 'low': 153.4,...",-3.004102,2025-10-17 13:52:34,0,0,0,2025-10-17 13:52:38,"{'buy': [{'quantity': 71, 'price': 153.64, 'or..."
1,True,full,194325505,153.69,3,155.10,4776310,682611,1070944,"{'open': 158.46, 'high': 158.68, 'low': 153.4,...",-3.004102,2025-10-17 13:52:39,0,0,0,2025-10-17 13:52:39,"{'buy': [{'quantity': 104, 'price': 153.66, 'o..."
2,True,full,194325505,153.69,3,155.10,4776310,682611,1070944,"{'open': 158.46, 'high': 158.68, 'low': 153.4,...",-3.004102,2025-10-17 13:52:39,0,0,0,2025-10-17 13:52:40,"{'buy': [{'quantity': 104, 'price': 153.66, 'o..."
3,True,full,194325505,153.69,11,155.10,4776310,682611,1070944,"{'open': 158.46, 'high': 158.68, 'low': 153.4,...",-3.004102,2025-10-17 13:52:39,0,0,0,2025-10-17 13:52:42,"{'buy': [{'quantity': 104, 'price': 153.66, 'o..."
4,True,full,194325505,153.68,1,155.10,4776310,682611,1070944,"{'open': 158.46, 'high': 158.68, 'low': 153.4,...",-3.010413,2025-10-17 13:52:39,0,0,0,2025-10-17 13:52:44,"{'buy': [{'quantity': 104, 'price': 153.66, 'o..."
5,True,full,194325505,153.68,1,155.10,4776575,682857,1071213,"{'open': 158.46, 'high': 158.68, 'low': 153.4,...",-3.010413,2025-10-17 13:52:44,0,0,0,2025-10-17 13:52:45,"{'buy': [{'quantity': 14, 'price': 153.66, 'or..."
6,True,full,194325505,153.68,35,155.10,4776575,682857,1071213,"{'open': 158.46, 'high': 158.68, 'low': 153.4,...",-3.010413,2025-10-17 13:52:44,0,0,0,2025-10-17 13:52:46,"{'buy': [{'quantity': 14, 'price': 153.66, 'or..."
7,True,full,194325505,153.67,79,155.09,4791694,674415,1071521,"{'open': 158.46, 'high': 158.68, 'low': 153.4,...",-3.016725,2025-10-17 13:52:50,0,0,0,2025-10-17 13:52:50,"{'buy': [{'quantity': 10, 'price': 153.59, 'or..."
8,True,full,194325505,153.68,5,155.09,4791694,674415,1071521,"{'open': 158.46, 'high': 158.68, 'low': 153.4,...",-3.010413,2025-10-17 13:52:50,0,0,0,2025-10-17 13:52:52,"{'buy': [{'quantity': 10, 'price': 153.59, 'or..."
9,True,full,194325505,153.52,181,155.09,4791694,674415,1071521,"{'open': 158.46, 'high': 158.68, 'low': 153.4,...",-3.111392,2025-10-17 13:52:50,0,0,0,2025-10-17 13:52:54,"{'buy': [{'quantity': 10, 'price': 153.59, 'or..."
